# Wireless Networking and Mobile Computing

### Prerequisites 
Run the code cell below once before starting. You can **collapse it afterwards**; the exercise cells below use the functions defined here.

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install numpy matplotlib ipywidgets --quiet

# --- Imports: libraries used in this notebook ---
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display


# --- Visualization helpers (no need to modify) ---

def _match_axes(ax1, ax2):
    ymin = min(ax1.get_ylim()[0], ax2.get_ylim()[0])
    ymax = max(ax1.get_ylim()[1], ax2.get_ylim()[1])
    xmin = min(ax1.get_xlim()[0], ax2.get_xlim()[0])
    xmax = max(ax1.get_xlim()[1], ax2.get_xlim()[1])

    for ax in (ax1, ax2):
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)


def plot_fft_vs_analytic(
    f_s, Xs_ct, Xa_short, f_l, Xl_ct, Xa_long
):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(
        f_s, np.abs(Xs_ct),
        label="|FFT approx| (short)", linewidth=2, color='yellow'
    )
    ax1.plot(
        f_s, np.abs(Xa_short),
        label="|analytic| (short)", linestyle="--", linewidth=2, color='orange'
    )
    ax1.set_xlabel("f [Hz]")
    ax1.set_ylabel("Magnitude")
    ax1.set_title("FFT vs Analytic (short pulse)")
    ax1.set_xlim([-15, 15])
    ax1.set_facecolor((0.2, 0.2, 0.3))
    ax1.grid(False)
    ax1.legend(fancybox=True, framealpha=1, loc="upper right",
               facecolor=(0.6, 0.61, 0.8))

    ax2.plot(
        f_l, np.abs(Xl_ct),
        label="|FFT approx| (long)", linewidth=2, color='yellow'
    )
    ax2.plot(
        f_l, np.abs(Xa_long),
        label="|analytic| (long)", linestyle="--", linewidth=2, color='orange'
    )
    ax2.set_xlabel("f [Hz]")
    ax2.set_ylabel("Magnitude")
    ax2.set_title("FFT vs Analytic (long pulse)")
    ax2.set_xlim([-15, 15])
    ax2.set_facecolor((0.2, 0.2, 0.3))
    ax2.grid(False)
    ax2.legend(fancybox=True, framealpha=1, loc="upper right",
               facecolor=(0.6, 0.61, 0.8))

    _match_axes(ax1, ax2)
    plt.tight_layout()
    plt.show()


def _plot_pulse_comparison(T1, T2, duration_max, frequency_limit):
    fig, axes = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)
    f = np.linspace(-frequency_limit, frequency_limit, 4001)

    for col, (T, color) in enumerate(((T1, 'yellow'), (T2, 'orange'))):
        time_ax, freq_ax = axes[:, col]

        # Exact edges avoid visually rounding the pulse duration to a time grid.
        edge = 2
        time_ax.plot(
            [-edge, -T/2, -T/2, T/2, T/2, edge],
            [0, 0, 1, 1, 0, 0],
            color=color, linewidth=2
        )
        time_ax.set(
            xlim=(-edge-0.2, edge+0.2), ylim=(-0.1, 1.2),
            xlabel='t [s]', ylabel='x(t)',
            title=f'Pulse {col+1}: T = {T:.2f} s'
        )

        freq_ax.plot(
            f, np.abs(X_analytic(f, T)),
            color=color, linewidth=2, label=f'|X(f)|, T = {T:g} s'
        )
        for zero in (-1/T, 1/T):
            freq_ax.axvline(
                zero, color='white', linestyle='--', linewidth=1
            )

        bracket_y = duration_max * 0.88
        freq_ax.annotate(
            '', xy=(1/T, bracket_y), xytext=(-1/T, bracket_y),
            arrowprops=dict(arrowstyle='<->', color='white')
        )
        freq_ax.text(
            0, bracket_y + duration_max * 0.05,
            f'Main-lobe width 2/T = {2/T:.2f} Hz',
            color='white', ha='center', va='bottom'
        )

        freq_ax.set(
            xlim=(-frequency_limit, frequency_limit),
            ylim=(0, duration_max * 1.1),
            xlabel='f [Hz]', ylabel='|X(f)|',
            title=f'First zeros: ±{1/T:.2f} Hz; peak: {T:.2f}'
        )
        freq_ax.legend(loc='upper right', fontsize=8, framealpha=0.6)

        for ax in (time_ax, freq_ax):
            ax.set_facecolor((0.2, 0.2, 0.3))
            ax.grid(False)

    plt.show()
    plt.close(fig)


def explore_pulse_durations(T_short, T_long):
    if min(T_short, T_long) <= 0:
        raise ValueError('Pulse durations must be positive.')
    # Include declared values even when students choose values outside the usual range.
    duration_min = min(0.1, T_short, T_long)
    duration_max = max(1.5, T_short, T_long)
    frequency_limit = 1.2 / duration_min
    controls = {}
    for key, label, value in (('T1', 'Pulse 1 T [s]', T_short),
                               ('T2', 'Pulse 2 T [s]', T_long)):
        controls[key] = widgets.FloatSlider(
            value=value, min=duration_min, max=duration_max, step=0.01,
            description=label, readout_format='.2f', continuous_update=True,
            style={'description_width': 'initial'}, layout=widgets.Layout(width='300px'))
    output = widgets.interactive_output(
        lambda T1, T2: _plot_pulse_comparison(T1, T2, duration_max, frequency_limit),
        controls)
    display(widgets.VBox([widgets.HBox(list(controls.values())), output]))
    return controls

### 1. Fourier Transform of a Rectangular Pulse

**Declarations**

Set the pulse durations below, then use **Run All**. The sliders initialize from these declarations.

- `T_short`: initial duration of Pulse 1 (Assignment 03: 0.2 s).
- `T_long`: initial duration of Pulse 2 (Assignment 03: 1.0 s).

The pulse labels remain valid even if you move the sliders until Pulse 1 is longer than Pulse 2. Slider exploration does not change the declarations; the FFT verification below uses the declared values.

In [ ]:
# --- Declarations: editable variables for students ---
T_short = ...
T_long  = ...

#### 1.1 Mathematical Explanation
Let the unit-height rectangular pulse of width $T>0$ be
$$
x(t) = \operatorname{rect}\!\left(\frac{t}{T}\right) =
\begin{cases}
1, & |t|<\tfrac{T}{2},\\
0, & \text{otherwise.}
\end{cases}
$$

Using the Fourier transform definition with frequency in Hertz (Hz)
$$
X(f) = \int_{-\infty}^{\infty} x(t)\,e^{-j2\pi f t}\,dt = \int_{-T/2}^{T/2} e^{-j2\pi f t}\,dt
= \left.\frac{e^{-j2\pi f t}}{-j2\pi f}\right|_{-T/2}^{T/2}
= \frac{e^{-j\pi f T}-e^{j\pi f T}}{-j2\pi f} = \frac{-2j\sin(\pi f T)}{-j2\pi f}
= T\,\frac{\sin(\pi f T)}{\pi f T}.
$$

Therefore, with the normalized $\operatorname{sinc}(x)=\dfrac{\sin(\pi x)}{\pi x}$,
$$
\boxed{\,X(f) = T\,\operatorname{sinc}(fT)\,}.
$$


The spectrum of the rectangular pulse has zeros at the frequencies $f = \pm \tfrac{1}{T}, \pm \tfrac{2}{T}, \dots$. The distance between the first negative and the first positive zero defines the main-lobe width, which is equal to $2/T$. If the pulse has an amplitude $A$ instead of unity, the entire spectrum is scaled accordingly, so the Fourier transform becomes $A \cdot X(f)$.



#### 1.2 General Integration Rule


$$
\begin{aligned}
&a \neq 0: \qquad \int e^{ax} dx = \frac{1}{a}\cdot e^{ax} + C, \qquad \text{ verified with } \quad (\frac{d}{dx}\left(\frac{1}{a}\cdot e^{ax}\right) = \frac{1}{a}\cdot \frac{d}{dx}\left(e^{ax}\right) = \frac{1}{a}\cdot a e^{ax} = e^{ax}) \\

&a=0: \qquad \int e^{0x} dx = \int 1 dx = x + C,
\end{aligned}
$$

where $C$ is the constant of integration. 

#### 1.3 Euler's Formula

The connection between the complex exponential function and sine/cosine is given by Euler’s formula:
$$
e^{j\theta} = \cos(\theta) + j\sin(\theta)
$$

where $e$ is the base of the natural logarithm, and $j = \sqrt{-1}$, and $\theta$ is a real angle (in radians). 

In engineering (especially in Electrical Engineering and Communications, to avoid confusion with electrical current $I$) we usually write $j$ instead of $i$. A cosine is the real part of a complex exponential.

### 2. Visualizing Time vs Frequency

In [ ]:
# --- Signal Definitions ---
def rect(t, T):
    """Unit-height rectangular pulse of width T centered at 0."""
    return (np.abs(t) < T/2).astype(float)

def X_analytic(f, T):
    """Analytic FT in Hz of rect(t/T): X(f)=T*sinc(fT) with normalized sinc."""
    x = f*T
    return T * np.sinc(x)

#### 2.1 Linked Time and Frequency Views


Move either duration slider and compare the pulse above with its spectrum below. Both columns use the same fixed axis scales. The dashed lines mark the first zeros, and the bracket shows their separation, $2/T$. 

In [ ]:
pulse_controls = explore_pulse_durations(T_short, T_long)

### 3. Numerical Verification (FFT)

We approximate the continuous-time pulse by sampling and compare its FFT magnitude against the analytic $|X(f)|$.

Note: This static check uses **the declared `T_short` and `T_long` values**, not the slider positions. To verify different durations, change the declarations and rerun the notebook.

In [ ]:
# --- FFT Analysis ---

# Parameters for FFT
Fs = 200.0
Ts = 1/Fs
N = 2**14
t_s = (np.arange(N) - N//2) * Ts

def sample_rect_and_fft(Tpulse):
    x = rect(t_s, Tpulse)
    X = np.fft.fftshift(np.fft.fft(np.fft.ifftshift(x)))
    f_s = (np.arange(N) - N//2) * Fs/N
    return f_s, X, x

f_s, Xs, xs = sample_rect_and_fft(T_short)
f_l, Xl, xl = sample_rect_and_fft(T_long)

# Scale the discrete FFT to approximate the continuous Fourier transform
Xs_ct = Ts * Xs
Xl_ct = Ts * Xl

Xa_short = X_analytic(f_s, T_short)
Xa_long = X_analytic(f_l, T_long)

plot_fft_vs_analytic(
    f_s, Xs_ct, Xa_short,
    f_l, Xl_ct, Xa_long
)
